> In case of autolabeing hier everything will be done

In [ ]:
#| default_exp labeling_agent

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
# | export
import os
from pathlib import Path
from tqdm.auto import tqdm
import shutil
import sys
# For data manipulation
import pandas as pd
import numpy as np

# For visualization
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import yaml
import tensorflow as tf
#import tf2onnx
#import onnxruntime as ort
import matplotlib as mpl
import json
import matplotlib as mpl

In [ ]:
#| export 
from fastcore.imports import *
from fastcore.foundation import *
from fastcore.all import *

In [ ]:
PRIVATE_EASY_PIN_DETECTION = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection')
sys.path.append(PRIVATE_EASY_PIN_DETECTION.as_posix())

In [ ]:

from cv_tools.core import *

In [ ]:
from private_easy_pin_detection.mask_generation_patch_image125 import *
#from labeling_test.seg_gpt import *

In [ ]:
mpl.rcParams['image.cmap'] = 'gray'

In [ ]:
def read_config(file_path):
    with open(file_path, 'r') as file:
        return yaml.safe_load(file)

In [ ]:
import os

In [ ]:
import os
os.environ['HF_HOME'] = '/home/ai_warstein/data/huggingface'
os.environ['XDG_CACHE_HOME'] = '/home/ai_warstein/data/huggingface'

In [ ]:
os.getenv('HF_HOME')

In [ ]:
hf_h = os.getenv('XDG_CACHE_HOME')

In [ ]:
hf_h

In [ ]:
SegGptForImageSegmentation?

In [ ]:
from transformers import SegGptImageProcessor, SegGptForImageSegmentation

checkpoint = "BAAI/seggpt-vit-large"
image_processor = SegGptImageProcessor.from_pretrained(checkpoint)
model = SegGptForImageSegmentation.from_pretrained(
    checkpoint, 
cache_dir=hf_h)
    #force_download=True,
    #resume_download=False)

In [ ]:
ref_msk_path= Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/sn_pin_masks')
ref_im_path= Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/sn_pin')
ref_dst = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/ref_images')
ref_dst.mkdir(exist_ok=True, parents=True)

In [ ]:
name_ = ref_msk_path.ls()[0].name
name_

In [ ]:
shutil.copyfile(Path(ref_im_path, name_),Path(ref_dst, name_))

In [ ]:
ref_im_path = Path(r'/home/ai_easypid.work/Reference_for_label_agent/pressfit_pin/prompt_images')

ref_msk_path = Path(r'/home/ai_easypid.work/Reference_for_label_agent/pressfit_pin/prompt_masks')
tst_im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/sn_pin')
tst_msk_path = Path(tst_im_path.parent, 'sn_pin_SegGPT_masks')
tst_msk_path.mkdir(exist_ok=True, parents=True)
tst_img = read_img(tst_im_path.ls()[0],cv=False).convert('RGB')


In [ ]:
ref_im_path = Path(r'/home/ai_easypid.work/Reference_for_label_agent/new_solder_pin/prompt_images')
ref_msk_path = Path(r'/home/ai_easypid.work/Reference_for_label_agent/new_solder_pin/prompt_masks')
test_im_path = Path(r'/home/ai_easypid.work/Reference_for_label_agent/new_solder_pin/test_images')
tst_msk_path = Path(r'/home/ai_easypid.work/Reference_for_label_agent/new_solder_pin/test_masks')
#ref_img = read_img(ref_im_path.ls()[0],cv=False)
tst_img_name = Path(test_im_path.ls()[0]).name
tst_img = read_img(test_im_path.ls()[0], cv=False, gray=False).convert('RGB')

In [ ]:
tst_im_path = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh9.h5/failed/missing/missing_pin_sn_images/')
tst_im_path.ls()

In [ ]:
tst_img_name = tst_im_path.ls()[0].name
tst_img = read_img(tst_im_path.ls()[0], cv=False, gray=False).convert('RGB')
tst_img

In [ ]:
M:\Fail_investigation\Missing_lead\missing_96_unzip\main_im2_cropped_masks\time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh9.h5\failed\missing\missing_pin_sn_images

In [ ]:
ref_img = read_img(ref_im_path.ls()[0], cv=False).convert('RGB')
name_ = Path(ref_im_path.ls()[0]).name
msk_path = Path(f'{ref_msk_path.ls()[0]}')
msk = read_img(msk_path, cv=True, gray=False)
ref_img

In [ ]:
msk.shape

In [ ]:
ref_img.mode

In [ ]:
ref_img.mode, tst_img.mode

In [ ]:
ref_img.size, tst_img.size

In [ ]:
#!pip show labeling_test

In [ ]:
test_image_mask = get_mask(
    image_prompt=ref_img,
    mask_prompt=msk,
    model=model,
    test_img=tst_img,
    save_msk_path=f'{tst_msk_path}/{tst_img_name}',
    show=True
)

In [ ]:
test_image_mask.shape

# PerSAM

In [ ]:
from transformers import AutoProcessor, SamModel

In [ ]:
processor = AutoProcessor.from_pretrained("facebook/sam-vit-huge")
# model = PerSamModel.from_pretrained("facebook/sam-vit-huge")
model = SamModel.from_pretrained("facebook/sam-vit-huge",cache_dir=hf_h)

In [ ]:
from labeling_test.os_persam import *

In [ ]:
outputs, tst_feat,topk_xy, topk_label, input_sam, best_idx = get_first_prediction(
                                        ref_img=ref_img,
                                        ref_msk=msk,
                                        tst_img=tst_img,
                                        model=model,
                                        processor=processor, 
                                        device='cpu',
                                        print_=False)

In [ ]:
show_(outputs['pred_masks'].to('cpu').numpy()[0][0][0])

In [ ]:
outputs_fr,_, _, _,masks_fr = get_first_refined_mask(
    ref_image=ref_img,
    ref_mask=msk,
    tst_img=tst_img,
    processor=processor,
    model=model,
    print_=False,
    device='cpu',
     )

In [ ]:
show_all_masks(masks_fr, outputs=outputs_fr)

In [ ]:
output_f, masks_f = get_last_refined_masks(
    ref_image=ref_img,
    ref_mask=msk,
    processor=processor,
    model=model,
    tst_img=tst_img,
    device='cpu',
    print_=False,
    outputs=outputs_fr,
    masks=masks_fr,
    best_idx=0
)

In [ ]:
show_all_masks(masks_f, outputs=output_f)

In [ ]:

masks_fr.shape

In [ ]:
np.transpose(np.array([test_image_mask, test_image_mask,test_image_mask]), (2, 1,0)).shape

In [ ]:
t_img = np.array([test_image_mask, test_image_mask,test_image_mask])
t_img.shape

In [ ]:
output_f, masks_f = get_last_refined_masks(
    ref_image=ref_img,
    ref_mask=msk,
    processor=processor,
    model=model,
    tst_img=tst_img,
    device='cpu',
    print_=False,
    outputs=outputs_fr,
    masks=t_img,
    best_idx=0
)

In [ ]:
show_all_masks(masks_f, outputs=output_f)

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/pressfit_image')
msk_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/missing_96_unzip/main_im2_cropped_masks/time_20_42_24_val_frGrnd0.9675_epoch_197_thrsh5.h5/passed/masks')
msk_path.ls()

In [ ]:
Path(im_path.parent, 'pressfit_masks')

In [ ]:
im_name = Path(im_path.ls()[0]).name
msk_fn = Path(msk_path, im_name)
msk  = read_img(msk_fn)

In [ ]:
shutil.copyfile(msk_fn, Path(im_path.parent, f'pressfit_masks/{im_name}'))

# Check missing pin with threshold 0.9 and threshold 0.5 difference 

In [ ]:
th_9_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/')

In [ ]:
config_path = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/config_office_test.yaml')
config = read_config(config_path)

In [ ]:
def process_image_im1_im2_j(
    im2:np.ndarray,
    im1:np.ndarray,
    c_first:bool=True
   )->[Tuple[np.ndarray, np.ndarray]]: # return two image of shape (None, 1152, 1632, 1)
    'process image 1 and image2'
    #if len(im1.shape) <3:
        #im1 = np.expand_dims(im1, axis=-1)
        #print(im1.shape)
    #if len(im2.shape) <3:
        #im2 = np.expand_dims(im2, axis=-1)
        #print(im2.shape)



    im1 = im1.astype(np.float32)/255.0
    im2 = im2.astype(np.float32)/255.0
    stack_img =  np.stack((im1, im2), axis=-1)[None,...]
    if c_first:
        return tf.transpose(stack_image, (0,3,1,2))
    return stack_img

In [ ]:
old_model = load_model(config)

# Creating New model from old model

In [ ]:
def create_new_model(
    model:tf.keras.Model,
    CHANNEL:int=2,
    HEIGHT:int=1152,
    WIDTH:int=1632
   ):
    'Create new model from old model'
    input_shape = (CHANNEL, HEIGHT, WIDTH)
    add_input = tf.keras.layers.Input(
        shape=input_shape, 
        name='InputImage'
        )
    x = tf.keras.layers.Permute((2,3, 1))(add_input)

    image1, image2 = x[:, :, :, 0:1], x[:, : ,: , 1:2]
    model_out = model(image2)

    model_out = tf.cast(model_out >= 0.9, tf.float32)
    output_ = tf.keras.layers.Multiply()([model_out, image1])
    return tf.keras.models.Model(inputs=[add_input], outputs=[output_])

In [ ]:
new_model = create_new_model(old_model)

# Preprocess and create prediction

In [ ]:
im2_path = Path(r'/home/ai_sintercra.work/VF_test_unzip/main_im2_cropped')
im1_path = Path(r'/home/ai_sintercra.work/VF_test_unzip/main_im1_cropped')

In [ ]:
s_fn = im2_path.ls()[0]
fn = s_fn.name
im1_fn = fn.replace('image2', 'image1')
im1_full_fn = f'{im1_path}/{im1_fn}'
im1 = read_img(im1_full_fn)
im2 = read_img(s_fn)

In [ ]:
stack_image = process_image_im1_im2_j(im2, im1)
stack_image.shape

In [ ]:
prediction_ =  new_model.predict(stack_image)

# Check overlay mask

In [ ]:
def pred_to_np255int8(msk:List[np.float32]):
    msk = msk[0]
    msk = msk * 255
    msk = msk.astype(np.uint8)
    return msk

In [ ]:
def overlay_mask(
                 img:Tuple[np.int8, np.int8, np.int8],
                 msk:Tuple[np.int8, np.int8],
                 show:bool=True
                 ):
    if not np.max(img)>1:
        img = img * 255
    img = img.astype(np.uint8)
    if len(img.shape) == 2 or img.shape[2] ==1:
       
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cntrs = find_contours_binary(msk)
    an_img = cv2.drawContours(img, cntrs, color= (0,255,0), contourIdx=-1, thickness=1)
    if show:

        show_(an_img)
    

In [ ]:
prd_msk = pred_to_np255int8(prediction_ > 0.1)
overlay_mask(img=im1, msk=prd_msk)

In [ ]:
prd_msk = pred_to_np255int8(prediction_ > 0.1)
overlay_mask(img=im2, msk=prd_msk)

In [ ]:
tst_im_path

In [ ]:
for idx, i in enumerate(tst_im_path.ls()):
    
    Path(i).rename(f'{tst_im_path}/image_{idx}.png')
    

In [ ]:
  v